In [2]:
!pip install undetected-chromedriver selenium pandas openpyxl requests

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 816.0 kB/s  0:00:16m0:00:0100:01
  Created wheel for undetected-chromedriver: filename=undetected_chromedriver-3.5.5-py3-none-any.whl size=47131 sha256=8764e79d52f2716850090e7f98669fe6f4a64989a33a2d93dba5c365d23fdc76
  Stored in directory: /home/pengguna/.cache/pip/wheels/91/52/80/bc5e4f27ed2d83197a1c14e3e560715e88883c1061cf3f47aa
Successfully built undetected-chromedriver
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [undetected-chromedriver]um]s]


In [1]:
import os
import time
import pandas as pd
import requests
import undetected_chromedriver as uc

# ================== PENGATURAN UTAMA ==================
TICKERS = ["TLKM", "BBCA"] 
DOWNLOAD_FILES = False # Ubah ke True jika ingin mendownload file PDF fisiknya
OUTPUT_DIR = "Data_IDX"
# ======================================================

def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

def download_file(url, folder_path, filename, cookies, user_agent):
    if not url: return
    file_path = os.path.join(folder_path, filename)
    headers = {
        "User-Agent": user_agent,
        "Referer": "https://www.idx.co.id/"
    }
    try:
        # Unduh file menggunakan tiket/cookie yang sudah tembus Cloudflare dari browser
        res = requests.get(url, headers=headers, cookies=cookies, stream=True)
        if res.status_code == 200:
            with open(file_path, 'wb') as f:
                for chunk in res.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"      [v] Berhasil unduh: {filename}")
        else:
            print(f"      [x] Gagal unduh {filename}, Status: {res.status_code}")
    except Exception as e:
        print(f"      [x] Error unduh {filename}: {e}")

def main():
    create_dir(OUTPUT_DIR)
    
    print("Membuka browser untuk melewati keamanan Cloudflare...")
    print("Harap jangan tutup browser yang muncul!")
    
    options = uc.ChromeOptions()
    # Sengaja tidak disembunyikan (headless) agar Cloudflare tidak curiga
    driver = uc.Chrome(options=options)
    
    # 1. Buka halaman utama web IDX dan tunggu hingga loading Cloudflare selesai
    driver.get("https://www.idx.co.id/")
    time.sleep(8) # Menunggu 8 detik agar sistem Cloudflare ("Just a moment") memvalidasi kita
    
    # Ekstrak 'tiket masuk' (Cookies & User-Agent) untuk fitur download file PDF nanti
    selenium_cookies = driver.get_cookies()
    session_cookies = {cookie['name']: cookie['value'] for cookie in selenium_cookies}
    user_agent = driver.execute_script("return navigator.userAgent;")
    
    for ticker in TICKERS:
        print(f"\n======================================")
        print(f"Mulai mengambil data Emiten: {ticker}")
        print(f"======================================")
        
        ticker_dir = os.path.join(OUTPUT_DIR, ticker)
        create_dir(ticker_dir)
        
        # -------------------------------------------------------------
        # SCRAPING DATA PROFIL DENGAN JAVASCRIPT BROWSER
        # -------------------------------------------------------------
        print(f"1. Memproses Data Profil & Dividen...")
        profile_url = f"https://www.idx.co.id/primary/ListedCompany/GetCompanyProfiles?emitenType=s&kodeEmiten={ticker}"
        
        # Kita menggunakan browser itu sendiri (lewat Javascript) untuk mengambil datanya
        fetch_script = f"""
            var callback = arguments[arguments.length - 1];
            fetch('{profile_url}')
            .then(response => response.json())
            .then(data => callback(data))
            .catch(err => callback({{"error": err.message}}));
        """
        
        try:
            driver.set_script_timeout(15)
            data = driver.execute_async_script(fetch_script)
            
            if data and "error" not in data:
                if "Profiles" in data and data["Profiles"]:
                    pd.DataFrame(data["Profiles"]).to_excel(os.path.join(ticker_dir, f"{ticker}_Profil.xlsx"), index=False)
                    print(f"   -> {ticker}_Profil.xlsx tersimpan.")
                    
                if "Dividend" in data and data["Dividend"]:
                    pd.DataFrame(data["Dividend"]).to_excel(os.path.join(ticker_dir, f"{ticker}_Dividen.xlsx"), index=False)
                    print(f"   -> {ticker}_Dividen.xlsx tersimpan.")
                    
                if "PemegangSaham" in data and data["PemegangSaham"]:
                    pd.DataFrame(data["PemegangSaham"]).to_excel(os.path.join(ticker_dir, f"{ticker}_PemegangSaham.xlsx"), index=False)
                    print(f"   -> {ticker}_PemegangSaham.xlsx tersimpan.")
            else:
                print(f"   [!] Gagal memuat data (kemungkinan nama emitmen salah/tidak ada).")
        except Exception as e:
            print(f"   [!] Error script profil: {e}")

        # -------------------------------------------------------------
        # SCRAPING LAPORAN KEUANGAN
        # -------------------------------------------------------------
        print(f"2. Memproses Laporan Keuangan...")
        finance_url = f"https://www.idx.co.id/primary/ListedCompany/GetFinancialReport?kodeEmiten={ticker}"
        
        fetch_finance = f"""
            var callback = arguments[arguments.length - 1];
            fetch('{finance_url}')
            .then(response => response.json())
            .then(data => callback(data))
            .catch(err => callback({{"error": err.message}}));
        """
        
        try:
            data_fin = driver.execute_async_script(fetch_finance)
            if data_fin and "Results" in data_fin:
                reports = data_fin.get("Results", [])
                if reports:
                    pd.DataFrame(reports).to_excel(os.path.join(ticker_dir, f"{ticker}_Daftar_Laporan.xlsx"), index=False)
                    print(f"   -> {ticker}_Daftar_Laporan.xlsx tersimpan.")
                    
                    collected_links = []
                    for rep in reports:
                        for att in rep.get("Attachments", []):
                            file_url = att.get("File_Path")
                            file_name = att.get("File_Name", f"{ticker}_Document.pdf")
                            
                            if file_url:
                                if file_url.startswith("/"):
                                    file_url = f"https://www.idx.co.id{file_url}"
                                collected_links.append(file_url)
                                
                                # Jika DOWNLOAD_FILES = True
                                if DOWNLOAD_FILES:
                                    download_file(file_url, ticker_dir, file_name, session_cookies, user_agent)

                    # Jika DOWNLOAD_FILES = False, simpan link-nya ke txt
                    if not DOWNLOAD_FILES and collected_links:
                        link_file = os.path.join(ticker_dir, f"{ticker}_Link_Dokumen.txt")
                        with open(link_file, "w") as f:
                            for link in collected_links:
                                f.write(f"{link}\n")
                        print(f"   -> Menyimpan {len(collected_links)} link dokumen ke {ticker}_Link_Dokumen.txt")
            else:
                print(f"   [!] Data laporan keuangan kosong/tidak tersedia.")
        except Exception as e:
             print(f"   [!] Error script laporan: {e}")

    # Menutup browser secara otomatis jika semua tugas sudah selesai
    driver.quit()
    print("\n======================================")
    print("PROSES SCRAPING SELESAI!")
    print(f"Semua data tersimpan di folder: {os.path.abspath(OUTPUT_DIR)}")

if __name__ == "__main__":
    main()

Membuka browser untuk melewati keamanan Cloudflare...
Harap jangan tutup browser yang muncul!


SessionNotCreatedException: Message: session not created: cannot connect to chrome at 127.0.0.1:35501
from session not created: This version of ChromeDriver only supports Chrome version 154
Current browser version is 153.0.8010.52; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x61c51c074e3a <unknown>
#1 0x61c51b99ee69 <unknown>
#2 0x61c51b9e59ee <unknown>
#3 0x61c51b9e47a0 <unknown>
#4 0x61c51b9d9897 <unknown>
#5 0x61c51ba2a58e <unknown>
#6 0x61c51ba29c4c <unknown>
#7 0x61c51b9e8748 <unknown>
#8 0x61c51b9e9511 <unknown>
#9 0x61c51c0373c5 <unknown>
#10 0x61c51c035bd5 <unknown>
#11 0x61c51c020c55 <unknown>
#12 0x61c51c03682a <unknown>
#13 0x61c51c009329 <unknown>
#14 0x61c51c05f248 <unknown>
#15 0x61c51c05f3e5 <unknown>
#16 0x61c51c073783 <unknown>
#17 0x7ee23f0a3dfa <unknown>
#18 0x7ee23f1375cc <unknown>
